# Experiment One: Caltech256 64x64 invariance evaluation
George Bird


In [1]:
from Dependencies import *
import torch.nn as nn
import torch
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchvision.datasets.utils import download_and_extract_archive
from torch.utils.data import DataLoader, Subset
import pandas as pd
import os

import copy
import pickle
from pathlib import Path


In [2]:
# Hyperparameters
LEARNING_RATE = 1e-3
BATCH_SIZE = 48
DEVICE = try_gpu(output=True, i=0)
ARCHITECTURE = [3 * 64 * 64, 100, 100, 257]
REPEATS = 20
PSI_DECAY = 1e-1
ADAMW_WEIGHT_DECAY = 1e-3
TOTAL_EPOCHS = 50

DATA_ROOT = Path("./data")
NORMALISATION = True
CALTECH256_IMAGE_SIZE = 64
CALTECH256_TRAIN_FRACTION = 0.8
CALTECH256_SPLIT_SEED = 20260504
CALTECH256_NORMALISATION_PATH = Path(
    f"./CALTECH256_{CALTECH256_IMAGE_SIZE}x{CALTECH256_IMAGE_SIZE}_normalisations.pkl"
)

save_dir = "./Saved_Models/Experiment 1/CALTECH256_64x64/Baseline/"
os.makedirs(save_dir, exist_ok=True)


GPU Name: NVIDIA GeForce RTX 3080
Memory Usage:
	Total:		 9.6 GB
	Allocated:	 0.0 GB
	Cached:		 0.0 GB



In [3]:
root = Path("data/caltech256/256_ObjectCategories")
categories = sorted(p.name for p in root.iterdir() if p.is_dir())

print(len(categories))
print(categories[74])
print(categories[82])

257
075.floppy-disk
083.gas-pump


In [4]:
# Caltech256 utilities copied narrowly from Experiment 6.
class RequiredPerPixelNormalize:
    """Use an existing per-pixel normalisation file; never create one."""

    def __init__(self, path):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(
                f"Required normalisation file not found: {path}. "
                "This experiment intentionally does not create normalisation .pkl files."
            )
        normaliser_dictionary = pkl.load(open(path, "rb"))
        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor):
        return (tensor - self.mean) * self.inv_std


CALTECH256_ARCHIVE_MD5 = "67b4f42ca05d46448c6bb8ecd2220f6d"
CALTECH256_DOWNLOAD_URLS = [
    "https://data.caltech.edu/records/nyy15-4j048/files/256_ObjectCategories.tar?download=1",
    "http://www.vision.caltech.edu/Image_Datasets/Caltech256/256_ObjectCategories.tar",
]


def ensure_caltech256_files_available():
    """Ensure Caltech256 is present without relying on torchvision's Google Drive URL."""
    caltech_root = DATA_ROOT / "caltech256"
    categories_dir = caltech_root / "256_ObjectCategories"
    if categories_dir.exists():
        return

    caltech_root.mkdir(parents=True, exist_ok=True)
    download_errors = []
    for url in CALTECH256_DOWNLOAD_URLS:
        try:
            download_and_extract_archive(
                url,
                download_root=str(caltech_root),
                filename="256_ObjectCategories.tar",
                md5=CALTECH256_ARCHIVE_MD5,
            )
            if categories_dir.exists():
                return
        except Exception as err:
            download_errors.append(f"{url}: {err}")

    raise RuntimeError(
        "Could not download/extract Caltech256 from any configured URL. "
        "You can also manually place 256_ObjectCategories under ./data/caltech256/.\n"
        + "\n".join(download_errors)
    )


def make_caltech256_transform():
    steps = [
        transforms.Lambda(lambda image: image.convert("RGB")),
        transforms.Resize((CALTECH256_IMAGE_SIZE, CALTECH256_IMAGE_SIZE)),
        transforms.ToTensor(),
    ]
    if NORMALISATION:
        steps.append(RequiredPerPixelNormalize(CALTECH256_NORMALISATION_PATH))
    return transforms.Compose(steps)


def make_caltech256_split_indices(dataset_length):
    rng = np.random.default_rng(CALTECH256_SPLIT_SEED)
    indices = rng.permutation(dataset_length)
    train_count = int(round(CALTECH256_TRAIN_FRACTION * dataset_length))
    return indices[:train_count].tolist(), indices[train_count:].tolist()


In [5]:
if NORMALISATION:
    print(f"Using normalisation: {CALTECH256_NORMALISATION_PATH}")
else:
    print("Not using normalisation")

ensure_caltech256_files_available()
transform = make_caltech256_transform()

caltech256_full = datasets.Caltech256(
    root=str(DATA_ROOT),
    download=False,
    transform=transform,
)
train_indices, test_indices = make_caltech256_split_indices(len(caltech256_full))
caltech256_train = Subset(caltech256_full, train_indices)
caltech256_test = Subset(caltech256_full, test_indices)

train_loader = DataLoader(caltech256_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(caltech256_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"Caltech256 64x64: train={len(caltech256_train)}, test={len(caltech256_test)}")


Using normalisation: CALTECH256_64x64_normalisations.pkl
Caltech256 64x64: train=3802, test=951


In [ ]:
combinations = [["NONE", "TRAINABLE+DECAY",], ["CONSTANT", "TRAINABLE+DECAY",], ["TRAINABLE", "TRAINABLE+DECAY",],
                ["TRAINABLE", "NONE"], ["TRAINABLE", "CONSTANT"], ["TRAINABLE", "TRAINABLE",], ["TRAINABLE", "EXPECTATION"]]

for REPEAT in range(REPEATS):
    for intrinsic_length_approach, linear_correction_approach in combinations:
        # Build filename
        save_name = f"repeat_{REPEAT:02d}_{intrinsic_length_approach}_{linear_correction_approach}_{TOTAL_EPOCHS}.pkl"
        save_path = os.path.join(save_dir, save_name)
        if save_name in os.listdir(save_dir): print(f"Skipping {save_name}"); continue
        else: print(f"Training {save_name}")


        network = IsotropicTanhMLP(
            layers = ARCHITECTURE,
            flatten = True, unflatten_shape = None,
            intrinsic_length_approach = intrinsic_length_approach,
            linear_correction_approach = linear_correction_approach,
            positive_intrinsic_length = True,
            init_intrinsic_length = 1e-6,
            tanh_epsilon = 1e-3,
            device = DEVICE, dtype = torch.get_default_dtype(),
            )

        # Initialise network parameters
        network.simple_initialiser(weight_init="orthogonal")

        # Train the baseline isotropic network once for classification on Caltech256 64x64
        optimiser = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE, weight_decay=ADAMW_WEIGHT_DECAY)
        trained_network, stats = training_loop(
            network=network,
            training_set=train_loader,
            testing_set=test_loader,
            epochs=TOTAL_EPOCHS,
            learning_rate=LEARNING_RATE,
            device=DEVICE,
            quiet=True,
            optimiser = optimiser,
            classification_or_reconstruction="classification",
            lambda_psi=PSI_DECAY if linear_correction_approach == "TRAINABLE+DECAY" else 0.0,
        )

        # Build filename
        save_name_pkl = f"repeat_{REPEAT:02d}_{intrinsic_length_approach}_{linear_correction_approach}_{TOTAL_EPOCHS}.pkl"
        save_name_svg = f"repeat_{REPEAT:02d}_{intrinsic_length_approach}_{linear_correction_approach}_{TOTAL_EPOCHS}.svg"
        save_path_pkl = os.path.join(save_dir, save_name_pkl)
        save_path_svg = os.path.join(save_dir, save_name_svg)

        train_x, test_x, train_cost, test_cost, train_acc, test_acc = stats



        train_x = np.asarray(train_x)
        train_acc = np.asarray(train_acc)
        test_x = np.asarray(test_x)
        test_acc = np.asarray(test_acc)

        # Average training accuracy within each epoch
        epoch_train_x = np.arange(1, TOTAL_EPOCHS + 1)
        epoch_train_acc = np.array([
            train_acc[(train_x >= epoch - 1) & (train_x < epoch)].mean()
            for epoch in epoch_train_x
        ])

        plt.figure(figsize=(8, 5))
        plt.plot(epoch_train_x, epoch_train_acc, label="Train Accuracy")
        plt.plot(test_x, test_acc, label="Test Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy (%)")
        plt.ylim(0, 100)
        plt.xlim(0, TOTAL_EPOCHS)
        plt.title("Caltech256 64x64 Accuracy vs Epoch")
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_path_svg)
        plt.show()



        # Save trained model and statistics

        with open(save_path_pkl, "wb") as f:
            pkl.dump({"stats": stats, "model": trained_network, }, f)

        print(f"Saved to {save_path}")


Skipping repeat_00_NONE_TRAINABLE+DECAY_50.pkl
Skipping repeat_00_CONSTANT_TRAINABLE+DECAY_50.pkl
Skipping repeat_00_TRAINABLE_TRAINABLE+DECAY_50.pkl
Skipping repeat_00_TRAINABLE_NONE_50.pkl
Skipping repeat_00_TRAINABLE_CONSTANT_50.pkl
Skipping repeat_00_TRAINABLE_TRAINABLE_50.pkl
Skipping repeat_00_TRAINABLE_EXPECTATION_50.pkl
Skipping repeat_01_NONE_TRAINABLE+DECAY_50.pkl
Skipping repeat_01_CONSTANT_TRAINABLE+DECAY_50.pkl
Skipping repeat_01_TRAINABLE_TRAINABLE+DECAY_50.pkl
Skipping repeat_01_TRAINABLE_NONE_50.pkl
Skipping repeat_01_TRAINABLE_CONSTANT_50.pkl
Skipping repeat_01_TRAINABLE_TRAINABLE_50.pkl
Skipping repeat_01_TRAINABLE_EXPECTATION_50.pkl
Skipping repeat_02_NONE_TRAINABLE+DECAY_50.pkl
Skipping repeat_02_CONSTANT_TRAINABLE+DECAY_50.pkl
Skipping repeat_02_TRAINABLE_TRAINABLE+DECAY_50.pkl
Skipping repeat_02_TRAINABLE_NONE_50.pkl
Skipping repeat_02_TRAINABLE_CONSTANT_50.pkl
Skipping repeat_02_TRAINABLE_TRAINABLE_50.pkl
Skipping repeat_02_TRAINABLE_EXPECTATION_50.pkl
Skipping r

Then undergo network resizing. Every row is evaluated from an independent clone of the same trained checkpoint; the existing full-diagonalisation-first resize procedure is unchanged.


In [ ]:
LAYER = 1   # For ARCHITECTURE=[12288, 100, 100, 257], full_diagonalise is only valid at layer=1.


ROWS = [+75, +50, +25, +5, +1, 0, -1, -5, -25, -50, -75, "baseline"]

COMBINATIONS = [
    ["NONE", "TRAINABLE+DECAY"],
    ["CONSTANT", "TRAINABLE+DECAY"],
    ["TRAINABLE", "TRAINABLE+DECAY"],
    ["TRAINABLE", "NONE"],
    ["TRAINABLE", "CONSTANT"],
    ["TRAINABLE", "TRAINABLE"],
    ["TRAINABLE", "EXPECTATION"],
]

SAVE_DIR = Path("./Saved_Models/Experiment 1/CALTECH256_64x64/Baseline/")


VERBOSE = True

# Output tex paths
OUTPUT_DIR = Path("./FinalResults/Exp1Results/Caltech256")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TEX = OUTPUT_DIR / "combined.tex"



# Helpers - Table formatting wrote by AI but checked myself, all data-sensitive parts written myself

def combo_to_key(combo):
    return f"{combo[0]}__{combo[1]}"

def combo_to_header(combo):
    return f"$o_{{\\mathrm{{{combo[0]}}}}}$, $\\psi_{{\\mathrm{{{combo[1]}}}}}$"

def safe_se(x):
    x = np.asarray(x, dtype=float)
    if x.size <= 1:
        return 0.0
    return np.std(x, ddof=1) / np.sqrt(x.size)

def format_cell(acc_mean, acc_se, scaled_eps_mean, scaled_eps_se,
                kl_mean, kl_se, sv_mean, sv_se, row_value,
                digits_acc=3, digits_scaled_eps=4,
                digits_kl=6, digits_sv=4):
    if row_value == "baseline":
        return (
            f"\\begin{{tabular}}[c]{{@{{}}c@{{}}}}"
            f"$a_0$ = {acc_mean:.{digits_acc}f} $\\pm$ {acc_se:.{digits_acc}f}\\% \\\\ "
            f"$\\sum \\sigma_0$ = {sv_mean:.{digits_sv}f} $\\pm$ {sv_se:.{digits_sv}f}"
            f"\\end{{tabular}}"
        )

    return (
        f"\\begin{{tabular}}[c]{{@{{}}c@{{}}}}"
        f"$\\Delta a$ = {acc_mean:.{digits_acc}f} $\\pm$ {acc_se:.{digits_acc}f} pp \\\\ "
        f"$\\Delta\\sum \\sigma$ = {sv_mean:.{digits_sv}f} $\\pm$ {sv_se:.{digits_sv}f} \\\\ "
        f"$\\epsilon_{{\\mathrm{{scaled}}}}$ = "
        f"{scaled_eps_mean:.{digits_scaled_eps}f} $\\pm$ {scaled_eps_se:.{digits_scaled_eps}f}\\% \\\\ "
        f"$D_{{\\mathrm{{KL}}}}$ = {kl_mean:.{digits_kl}f} $\\pm$ {kl_se:.{digits_kl}f} nats"
        f"\\end{{tabular}}"
    )

def clone_model(model):
    m = copy.deepcopy(model)
    m.eval()
    return m

@torch.no_grad()
def predict_logits(model, dataloader, device):
    model.eval()
    ys = []
    yhat = []
    for xb, yb in dataloader:
        xb = xb.to(device)
        out = model(xb)
        yhat.append(out.detach().cpu())
        ys.append(yb.detach().cpu())
    return torch.cat(yhat, dim=0), torch.cat(ys, dim=0)

def accuracy_from_logits(logits, labels):
    pred = logits.argmax(dim=1)
    return 100.0 * (pred == labels).float().mean().item()

def scaled_epsilon_from_logits(logits_after, logits_before):
    # 100/N sum_i ||z_post(i)-z_pre(i)||_2 / ||z_pre(i)||_2
    before_norm = torch.linalg.norm(logits_before, dim=1)
    zero_norm = before_norm == 0
    if torch.any(zero_norm):
        raise ZeroDivisionError(
            "Divide-by zero in scale-aware epsilon probably logits having zero L2 norm"
        )
    diff_norm = torch.linalg.norm(logits_after - logits_before, dim=1)
    return (100.0 * diff_norm / before_norm).mean().item()

def kl_divergence_from_logits(logits_after, logits_before):
    # Mean D_KL(softmax(z_pre) || softmax(z_post)), reported in nats.
    p_before = torch.softmax(logits_before, dim=1)
    log_p_after = torch.log_softmax(logits_after, dim=1)
    return torch.nn.functional.kl_div(
        log_p_after,
        p_before,
        reduction="batchmean",
    ).item()

def singular_value_sum(model, layer):
    singular_values = model.get_singular_values(layer=layer, use_torch=True)
    return float(singular_values.detach().cpu().sum().item())

def apply_resize(model, layer, row_value):
    """
    baseline: untouched
    0: full diagonalise only
    +k: full diagonalise then k neurogenerate
    -k: full diagonalise then k neurodegenerate
    """
    resized = clone_model(model)

    if row_value == "baseline":
        return resized

    resized.full_diagonalise(layer)

    if row_value > 0:
        for _ in range(int(row_value)):
            resized.neurogenerate(layer)
    elif row_value < 0:
        for _ in range(int(abs(row_value))):
            resized.neurodegenerate(layer)

    return resized

def load_saved_model(combo, repeat_idx):
    intrinsic, correction = combo
    path = SAVE_DIR / f"repeat_{repeat_idx:02d}_{intrinsic}_{correction}_{TOTAL_EPOCHS}.pkl"

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find saved pickle:\n  {path}\n"
        )

    with open(path, "rb") as f:
        obj = pickle.load(f)

    # Expected format: {'stats': ..., 'model': ...}
    if isinstance(obj, dict) and "model" in obj:
        model = obj["model"]
        stats = obj.get("stats", None)
    else:
        model = obj
        stats = None

    return model, stats, path

def make_latex_table(df, caption=None, label=None):
    headers = list(df.columns)
    col_spec = "l" + "c" * len(headers)

    lines = []
    lines.append("\\begin{table}[ht]")
    lines.append("\\centering")
    lines.append("\\scriptsize")
    lines.append(f"\\begin{{tabular}}{{{col_spec}}}")
    lines.append("\\hline")
    lines.append("Row & " + " & ".join(headers) + " \\\\")
    lines.append("\\hline")

    for idx, row in df.iterrows():
        cells = [str(idx)] + [row[h] for h in headers]
        lines.append(" & ".join(cells) + " \\\\")

    lines.append("\\hline")
    lines.append("\\end{tabular}")

    if caption is not None:
        lines.append(f"\\caption{{{caption}}}")
    if label is not None:
        lines.append(f"\\label{{{label}}}")

    lines.append("\\end{table}")
    return "\n".join(lines)


# Main evaluation

device = try_gpu()


if not SAVE_DIR.exists():
    raise FileNotFoundError(f"SAVE_DIR does not exist: {SAVE_DIR.resolve()}")

results_nested = {}
paths_used = {}

for combo in COMBINATIONS:
    combo_key = combo_to_key(combo)
    results_nested[combo_key] = {}

    if VERBOSE:
        print(f"\n=== Combination: {combo} ===")

    # Per row, store paired per-repeat statistics
    per_row_acc_values = {row: [] for row in ROWS}
    per_row_scaled_eps_values = {row: [] for row in ROWS}
    per_row_kl_values = {row: [] for row in ROWS}
    per_row_sv_values = {row: [] for row in ROWS}

    for repeat_idx in range(REPEATS):
        base_model, base_stats, base_path = load_saved_model(combo, repeat_idx)
        paths_used[(combo_key, repeat_idx)] = str(base_path)

        base_model = clone_model(base_model).to(device)
        base_logits, base_labels = predict_logits(base_model, test_loader, device)
        base_acc = accuracy_from_logits(base_logits, base_labels)
        base_sv_sum = singular_value_sum(base_model, LAYER)

        if VERBOSE:
            print(
                f"repeat {repeat_idx:02d} | baseline acc = {base_acc:.4f}% "
                f"| baseline sum(sigma) = {base_sv_sum:.4f} | {base_path}"
            )

        for row in ROWS:
            resized_model = apply_resize(base_model, LAYER, row).to(device)
            resized_logits, resized_labels = predict_logits(resized_model, test_loader, device)
            resized_sv_sum = singular_value_sum(resized_model, LAYER)

            # Sanity: labels should match
            if not torch.equal(base_labels, resized_labels):
                raise RuntimeError("Test labels changed between baseline and resized evaluation.")

            resized_acc = accuracy_from_logits(resized_logits, resized_labels)

            if row == "baseline":
                # Baseline reports raw accuracy and original sum of singular values.
                acc_value = base_acc
                scaled_eps_value = np.nan
                kl_value = np.nan
                sv_value = base_sv_sum
            else:
                # All paired function-movement metrics use the same pre/post logits.
                acc_value = resized_acc - base_acc
                scaled_eps_value = scaled_epsilon_from_logits(resized_logits, base_logits)
                kl_value = kl_divergence_from_logits(resized_logits, base_logits)
                sv_value = base_sv_sum - resized_sv_sum

            per_row_acc_values[row].append(acc_value)
            per_row_scaled_eps_values[row].append(scaled_eps_value)
            per_row_kl_values[row].append(kl_value)
            per_row_sv_values[row].append(sv_value)

            del resized_model, resized_logits, resized_labels
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del base_model, base_logits, base_labels
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Aggregate after all repeats for this combo
    for row in ROWS:
        acc_arr = np.asarray(per_row_acc_values[row], dtype=float)
        scaled_eps_arr = np.asarray(per_row_scaled_eps_values[row], dtype=float)
        kl_arr = np.asarray(per_row_kl_values[row], dtype=float)
        sv_arr = np.asarray(per_row_sv_values[row], dtype=float)

        results_nested[combo_key][row] = {
            "acc_mean": float(np.mean(acc_arr)),
            "acc_se": float(safe_se(acc_arr)),
            "scaled_eps_mean": float(np.nanmean(scaled_eps_arr)) if not np.all(np.isnan(scaled_eps_arr)) else np.nan,
            "scaled_eps_se": float(safe_se(scaled_eps_arr[~np.isnan(scaled_eps_arr)])) if not np.all(np.isnan(scaled_eps_arr)) else np.nan,
            "kl_mean": float(np.nanmean(kl_arr)) if not np.all(np.isnan(kl_arr)) else np.nan,
            "kl_se": float(safe_se(kl_arr[~np.isnan(kl_arr)])) if not np.all(np.isnan(kl_arr)) else np.nan,
            "sv_mean": float(np.mean(sv_arr)),
            "sv_se": float(safe_se(sv_arr)),
            "acc_values": acc_arr,
            "scaled_eps_values": scaled_eps_arr,
            "kl_values": kl_arr,
            "sv_values": sv_arr,
        }


# -------------------------
# Build display DataFrames
# -------------------------
accuracy_df = pd.DataFrame(index=[str(r) for r in ROWS])
sv_df = pd.DataFrame(index=[str(r) for r in ROWS])
scaled_eps_df = pd.DataFrame(index=[str(r) for r in ROWS])
kl_df = pd.DataFrame(index=[str(r) for r in ROWS])
display_df = pd.DataFrame(index=[str(r) for r in ROWS])

for combo in COMBINATIONS:
    combo_key = combo_to_key(combo)
    header = combo_to_header(combo)

    accuracy_cells = []
    sv_cells = []
    scaled_eps_cells = []
    kl_cells = []
    combined_cells = []

    for row in ROWS:
        stats = results_nested[combo_key][row]

        if row == "baseline":
            accuracy_cells.append(
                f"$a_0$ = {stats['acc_mean']:.3f} $\\pm$ {stats['acc_se']:.3f}\\%"
            )
            sv_cells.append(
                f"$\\sum \\sigma_0$ = {stats['sv_mean']:.4f} $\\pm$ {stats['sv_se']:.4f}"
            )
            scaled_eps_cells.append("--")
            kl_cells.append("--")
        else:
            accuracy_cells.append(
                f"{stats['acc_mean']:.3f} $\\pm$ {stats['acc_se']:.3f} pp"
            )
            sv_cells.append(
                f"{stats['sv_mean']:.4f} $\\pm$ {stats['sv_se']:.4f}"
            )
            scaled_eps_cells.append(
                f"{stats['scaled_eps_mean']:.4f} $\\pm$ {stats['scaled_eps_se']:.4f}\\%"
            )
            kl_cells.append(
                f"{stats['kl_mean']:.6f} $\\pm$ {stats['kl_se']:.6f}"
            )

        combined_cells.append(
            format_cell(
                stats["acc_mean"], stats["acc_se"],
                stats["scaled_eps_mean"], stats["scaled_eps_se"],
                stats["kl_mean"], stats["kl_se"],
                stats["sv_mean"], stats["sv_se"],
                row,
                digits_acc=3, digits_scaled_eps=4,
                digits_kl=6, digits_sv=4
            )
        )

    accuracy_df[header] = accuracy_cells
    sv_df[header] = sv_cells
    scaled_eps_df[header] = scaled_eps_cells
    kl_df[header] = kl_cells
    display_df[header] = combined_cells

display(display_df)


# -------------------------
# Write LaTeX
# -------------------------
caption_common = (
    f"Caltech256 $64\\times64$ resize evaluation for the MLP architecture "
    f"$12288$--$100$--$100$--$257$ (two hidden layers of width 100), resizing at layer $L={LAYER}$. "
    f"Every non-baseline row first applies full diagonalisation; $+k$ then applies $k$ neurogeneration "
    f"operations, $-k$ applies $k$ neurodegeneration operations, and $0$ is diagonalisation only. "
    f"All reported uncertainties are mean $\\pm$ standard error over {REPEATS} repeats. "
)

captions = {
    "accuracy_change.tex": (
        caption_common
        + f"For non-baseline rows, $\\Delta a=a_\\mathrm{{post}}-a_\\mathrm{{pre}}$ is reported in "
        f"percentage points (pp), so positive values indicate increased accuracy. "
        f"The baseline row reports the raw test accuracy $a_0$ in percent."
    ),
    "singular_value_change.tex": (
        caption_common
        + f"For non-baseline rows, $\\Delta\\sum\\sigma=\\sum\\sigma_\\mathrm{{pre}}-"
        f"\\sum\\sigma_\\mathrm{{post}}$, so positive values indicate singular-value mass removed. "
        f"The baseline row reports the original $\\sum\\sigma_0$."
    ),
    "scaled_epsilon.tex": (
        caption_common
        + f"Scale-aware epsilon is "
        f"$\\epsilon_\\mathrm{{scaled}}=\\frac{{100}}{{N}}\\sum_i "
        f"\\frac{{\\lVert z_\\mathrm{{post}}^{{(i)}}-z_\\mathrm{{pre}}^{{(i)}}\\rVert_2}}"
        f"{{\\lVert z_\\mathrm{{pre}}^{{(i)}}\\rVert_2}}$ and is reported as a percentage. "
        f"The baseline entry is not applicable."
    ),
    "kl_divergence.tex": (
        caption_common
        + f"KL divergence is "
        f"$D_\\mathrm{{KL}}=\\frac{{1}}{{N}}\\sum_i "
        f"D_\\mathrm{{KL}}(\\operatorname{{softmax}}(z_\\mathrm{{pre}}^{{(i)}})"
        f"\\,\\|\\,\\operatorname{{softmax}}(z_\\mathrm{{post}}^{{(i)}}))$ and is reported in nats. "
        f"The baseline entry is not applicable."
    ),
    "combined.tex": (
        caption_common
        + f"For non-baseline rows, $\\Delta a=a_\\mathrm{{post}}-a_\\mathrm{{pre}}$ is in percentage "
        f"points; $\\Delta\\sum\\sigma=\\sum\\sigma_\\mathrm{{pre}}-\\sum\\sigma_\\mathrm{{post}}$, "
        f"so positive values indicate singular-value mass removed; "
        f"$\\epsilon_\\mathrm{{scaled}}=\\frac{{100}}{{N}}\\sum_i "
        f"\\frac{{\\lVert z_\\mathrm{{post}}^{{(i)}}-z_\\mathrm{{pre}}^{{(i)}}\\rVert_2}}"
        f"{{\\lVert z_\\mathrm{{pre}}^{{(i)}}\\rVert_2}}$ is a percentage; and "
        f"$D_\\mathrm{{KL}}=\\frac{{1}}{{N}}\\sum_i "
        f"D_\\mathrm{{KL}}(\\operatorname{{softmax}}(z_\\mathrm{{pre}}^{{(i)}})"
        f"\\,\\|\\,\\operatorname{{softmax}}(z_\\mathrm{{post}}^{{(i)}}))$ is in nats. "
        f"The baseline row reports raw $a_0$ and $\\sum\\sigma_0$."
    ),
}

tables = {
    "accuracy_change.tex": (accuracy_df, "tab:caltech256_accuracy_change"),
    "singular_value_change.tex": (sv_df, "tab:caltech256_singular_value_change"),
    "scaled_epsilon.tex": (scaled_eps_df, "tab:caltech256_scaled_epsilon"),
    "kl_divergence.tex": (kl_df, "tab:caltech256_kl_divergence"),
    "combined.tex": (display_df, "tab:caltech256_combined"),
}

for filename, (table_df, label) in tables.items():
    latex = make_latex_table(table_df, caption=captions[filename], label=label)
    output_path = OUTPUT_DIR / filename
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(latex)
    print("Wrote LaTeX table to:", output_path)

latex_table = make_latex_table(
    display_df,
    caption=captions["combined.tex"],
    label="tab:caltech256_combined",
)


# -------------------------
# Optional: expose raw numeric summaries too
# -------------------------
raw_rows = []
for combo in COMBINATIONS:
    combo_key = combo_to_key(combo)
    for row in ROWS:
        s = results_nested[combo_key][row]
        raw_rows.append({
            "combination": combo_to_header(combo),
            "row": row,
            "acc_mean": s["acc_mean"],
            "acc_se": s["acc_se"],
            "scaled_eps_percent_mean": s["scaled_eps_mean"],
            "scaled_eps_percent_se": s["scaled_eps_se"],
            "kl_nats_mean": s["kl_mean"],
            "kl_nats_se": s["kl_se"],
            "sv_removed_or_baseline_sum_mean": s["sv_mean"],
            "sv_removed_or_baseline_sum_se": s["sv_se"],
        })

raw_df = pd.DataFrame(raw_rows)
display(raw_df)
